# Feature engineering

This notebook builds our engineered features, chooses which raw columns go into the models, and saves everything
to `data/processed/features_v2.parquet` with the column lists in `config/features_v2.yaml`, so a model notebook
can load them directly.

The feature functions are in `src/domain_features.py` and the column-selection functions in
`src/feature_selection.py`. This notebook calls them, explains the reasoning and the EDA evidence behind each
group of features, and checks them on the training block. Section numbers like "EDA 9.4" refer to `01_eda.ipynb`.

Each engineered feature comes from one of two places:

- **EDA:** a pattern found in the EDA (for example, D columns that count days from a fixed date).
- **Domain knowledge:** a card-fraud red flag that the EDA tested in its section 9. Some were supported. For
  others the expected direction didn't hold (for example, a known customer on a new device has *less* fraud), but
  the EDA still showed a stable difference, so the feature is kept as a signal, not as the original red flag.

Every engineered feature also has to hold over time and have EDA support: 21 candidate features were left out
because their link to fraud reversed, faded or never showed between the first and last 30 days of the training
block, or because the EDA found no support for the idea behind them or their signal is already carried by a kept
feature (section 10). The raw columns
used as model inputs are chosen in section 11, following EDA section 13.

Rules we followed for every feature:

- Features that describe a customer's history only look at earlier transactions. The table is sorted by
  `TransactionDT` then `TransactionID`, so "earlier" means earlier in that order. This matches what a live system
  would know when a transaction comes in. `tests/test_domain_features.py` checks this.
- Anything learned from the data (the most common `addr2` and `card3` values, the V-column groups, which columns
  to keep) comes from the training block only, i.e. the first 80% of rows by time.
- No feature uses `isFraud`.
- Imputation, scaling and encoding are not done here. They belong in the model notebook and have to be fit on
  training rows only.

In [1]:
import sys; sys.path.insert(0, '..')
import time
import numpy as np
import pandas as pd
import yaml
from sklearn.metrics import roc_auc_score

from src.data import load_joined
from src import domain_features as dfe
from src import feature_selection as fsel

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 30)

t0 = time.time()
df = load_joined()                       # time-sorted joined table from src/data.py
assert df['TransactionDT'].is_monotonic_increasing
train = df.iloc[:dfe.N_TRAIN]            # training block: first 80% by time
y_train = train['isFraud'].to_numpy()
print(df.shape, f'train block rows = {len(train):,}, train fraud rate = {y_train.mean():.4f}')

(590540, 436) train block rows = 472,432, train fraud rate = 0.0351


### Quick check

For each new feature we look at the training block only and report how much of it is missing, plus the AUROC
you get from that feature on its own (flipped if it's below 0.5, so 0.5 means no signal either way). For 0/1
flags we also show the fraud rate in each group.

This is just to see whether a feature behaves the way we expected. It isn't how we decide what to keep: that is
the time-stability check in section 10 together with the EDA evidence, and the models' time-based CV comes after.
A feature can be weak on its own and still help a model.

In [2]:
def signal_table(frame):
    rows = []
    for c in frame.columns:
        s = frame[c].iloc[:dfe.N_TRAIN]
        row = {'feature': c, 'pct_missing': 100 * s.isna().mean()}
        if isinstance(s.dtype, pd.CategoricalDtype):
            row['n_levels'] = s.nunique()
        else:
            v = s.to_numpy(np.float64); m = ~np.isnan(v)
            if m.sum() and len(np.unique(y_train[m])) > 1 and len(np.unique(v[m])) > 1:
                auc = roc_auc_score(y_train[m], v[m]); row['auroc_1feat'] = max(auc, 1 - auc)
            if set(np.unique(v[m])) <= {0.0, 1.0}:
                row['fraud_rate_if_1'] = y_train[m][v[m] == 1].mean() if (v[m] == 1).any() else np.nan
                row['fraud_rate_if_0'] = y_train[m][v[m] == 0].mean() if (v[m] == 0).any() else np.nan
                row['pct_rows_1'] = 100 * (v[m] == 1).mean()
        rows.append(row)
    return pd.DataFrame(rows).set_index('feature').round(4)

def rate_by_level(frame, col, top=10):
    s = frame[col].iloc[:dfe.N_TRAIN].astype(object).fillna('<missing>')
    t = pd.DataFrame({'level': s, 'isFraud': y_train}).groupby('level')['isFraud'].agg(n='size', fraud_rate='mean')
    return t.sort_values('n', ascending=False).head(top).round(4)

## 0. Training-block constants and the customer ID

A couple of the flags below compare a value with the most common one, so we first need to know what's common.
We take the most common `addr2` (looks like a country code) and `card3` from the training block. We also take the
V-column groups from EDA section 3, i.e. columns that are missing on exactly the same rows.

Vesta marks a customer's later transactions as fraud once one of them is reported, so fraud tends to stick to a
customer rather than to a single transaction. The data has no customer ID, so following `Context.md` we
approximate one:

`uid = card1 _ addr1 _ floor(TransactionDay − D1)`

`card1` roughly identifies the card, `addr1` is the billing region, and `TransactionDay − D1` estimates the day
the account was opened, which stays the same for a given customer. EDA 5.1 compares this key with coarser and
stricter alternatives.

We use `uid` to group transactions when building the history features. It's saved in the output file so it can
be used for grouped CV or the returning vs new customer analysis, but it shouldn't be given to a model as a
feature, because the model would just memorise the IDs.

In [3]:
consts = dfe.fit_constants(train)
print({k: v for k, v in consts.items() if k != 'v_group_representatives'})
print('V-group representative columns:', consts['v_group_representatives'])

uid, fams = dfe.build_features(df, consts)
print(f'built {sum(f.shape[1] for f in fams.values())} features in {time.time() - t0:.1f}s')

sizes = uid.iloc[:dfe.N_TRAIN].value_counts()
sizes = sizes[sizes > 0]   # drop uid categories that only occur in later blocks
print(f"train-block uids: {len(sizes):,}; median group size {sizes.median():.0f}; "
      f"share of rows in single-transaction uids {100 * (sizes == 1).sum() / dfe.N_TRAIN:.1f}%")

{'addr2_mode': 87.0, 'card3_mode': 150.0, 'n_train_rows': 472432}
V-group representative columns: ['V279', 'V95', 'V281', 'V12', 'V53', 'V75', 'V35', 'V1', 'V220', 'V169', 'V167', 'V217', 'V322', 'V138']
built 55 features in 12.1s
train-block uids: 182,535; median group size 1; share of rows in single-transaction uids 22.5%


## 1. Transaction amount

| Feature | What it is | Source | Why we think it helps |
| --- | --- | --- | --- |
| `fe_amt_log` | log(1 + amount) | EDA 2.3, 7 | Amounts are very skewed (mean \$134, 75th percentile \$125, maximum \$31,937). The log adds no new information (EDA 7: same order, U-shaped fraud rate), but LR and the MLP work better on a log scale; trees ignore it. |
| `fe_amt_cents` | the cents part (0 to 99) | EDA 7 | Endings differ a lot: `.95` has 0.7% fraud, `.00` 3.6%, uncommon endings 9.5%. |
| `fe_amt_decimals` | number of decimal places (capped at 3) | EDA 7 | Three or more decimals usually means the amount was converted from another currency, so the purchase is probably cross-border. EDA 7: more than two decimals has 11.3% fraud vs 1.1% for exact cents. |
| `fe_amt_lt5` | amount under $5 | Domain knowledge, EDA 9.3 (supported, rare) | Stolen cards are often tested with a small purchase first. Under $5 has 6.5% fraud, but only 920 training rows. |

These only use the transaction's own amount, so there's no leakage risk. Considered but left out:
`fe_amt_round_dollar` (whole-dollar amount; faded) and `fe_amt_round_10` (multiple of 10; reversed); see section 10.

In [4]:
signal_table(fams['amount'])

,pct_missing,auroc_1feat,fraud_rate_if_1,fraud_rate_if_0,pct_rows_1
feature,,,,,
fe_amt_log,0.0,0.5019,NaN,NaN,NaN
fe_amt_cents,0.0,0.5606,NaN,NaN,NaN
fe_amt_decimals,0.0,0.5503,NaN,NaN,NaN
fe_amt_lt5,0.0,0.5009,0.0652,0.0351,0.1947


## 2. Time of day and customer habits

| Feature | What it is | Source | Why we think it helps |
| --- | --- | --- | --- |
| `fe_hour` | hour of day | EDA 4 | We don't know the real start time of `TransactionDT`, so the hour is relative, but the daily pattern is strong: the quiet hours 6 to 10 have 6.9–10.6% fraud vs 2.1–3.9% in busy hours. |
| `fe_uid_hour_dev` | how many hours this transaction is from the customer's usual time | Domain knowledge, EDA 9.9 (direction not supported) | The red flag was "an unusual hour is riskier"; the EDA found the opposite, but a weak, stable difference. Measured around the clock, so 23:00 and 01:00 are 2 hours apart. |

The customer's usual hour is worked out from their earlier transactions only, so it's missing for their first one.
Considered but left out: `fe_uid_hour_concentration` (how regular the customer's shopping hour is; faded) and
`fe_dow` (day of week; no clear signal in either window, and EDA 9.8 found no day-of-week effect).

In [5]:
signal_table(fams['time'])

,pct_missing,auroc_1feat
feature,,
fe_hour,0.0000,0.5093
fe_uid_hour_dev,38.6373,0.5245


## 3. Normalised D columns

Source: EDA 7.1 and 7.2. Several D columns count days since some event, so they keep going up over time for every
customer and the raw value partly stands in for the date. EDA 7.2 checked all 15 D columns: D1, D4, D10, D11 and
D15 drift about one day per day, and after subtracting them from the transaction day the value is constant for
65–100% of customers instead of 22–36%. Subtracting gives a fixed date for each customer:

`fe_Dxn = TransactionDay − Dx` for D1, D4, D10, D11 and D15

D2 repeats D1, and the other D columns don't count from a fixed date or are about 90% missing.

On their own these are no more stable over time than the raw D columns (EDA 11.1), so they are **added** next to
the raw columns rather than replacing them; a tree model can still combine them with other columns. EDA 8.1
found that disagreement between a customer's dates is not a stable fraud signal on its own, and EDA 11.1 that their
main value is in identifying customers (D1's version is the account start in `uid`). Negative values just mean the account was opened before the data starts. Only the
row's own values are used, so there's no leakage.

In [6]:
signal_table(fams['d_norm'])

,pct_missing,auroc_1feat
feature,,
fe_D1n,0.1981,0.6272
fe_D4n,29.5861,0.6050
fe_D10n,14.0530,0.6277
fe_D11n,52.0653,0.5948
fe_D15n,16.4606,0.6348


## 4. First-time use of a device, email, etc.

Source: domain knowledge, tested in EDA 9.1, 9.2 and 9.12. When an account is taken over, the fraudster
usually has the card details but not the owner's phone, browser or email address, so the red flag is a customer
suddenly using a device or email they've never used before.

| Feature | What it is | EDA result |
| --- | --- | --- |
| `fe_uid_new_{device, browser, email, product}` | 1 if the customer has used other values before but never this one, 0 if they've used this one before. Missing if there's nothing to compare with. | 9.1: direction not supported. A new value is *less* risky than a known one: device 7.1% vs 9.6%, browser 7.9% vs 10.6%, product 2.4% vs 4.1%, email 3.7% vs 4.3% (small). Kept as a stable signal. |
| `fe_uid_n_prior_{devices, emails, browsers}` | How many different values the customer has used so far | 9.12: supported |

We also look at it from the device side. Fraud rings often push many stolen cards through the same device, so we
build a rough device fingerprint from `DeviceInfo`, OS, browser and screen resolution, and count (EDA 9.2:
supported, 4.5% fraud with no earlier cards on the device vs 11.5% with 2–5; customers and transactions per device
rise and fall the same way):

| Feature | What it is |
| --- | --- |
| `fe_fp_n_prior_cards` | different cards seen on this fingerprint so far |
| `fe_fp_n_prior_uids` | different customers seen on it so far |
| `fe_fp_prior_count` | number of earlier transactions from it |

These are missing when there's no device information, which is the case for most rows without an identity record.
All the counts in this section only use earlier rows. Considered but left out: `fe_uid_new_os` and
`fe_uid_new_screen` (no clear signal in either window, and built from the OS and screen fields whose raw values are
excluded), and the same idea at the card level: `fe_card1_new_addr1`, `fe_card1_n_prior_addr1` and
`fe_card1_n_prior_emails`. EDA 9.11 found no support for these: a new billing region for a `card1` has the same
fraud rate as a known one (2.5% each), and the counts show no steady rise, because one `card1` value is shared by many
honest customers. `fe_card1_new_addr1` only separates fraud through its missing state, which mostly repeats missing
`addr1` (plus each card's first transaction).

In [7]:
signal_table(fams['novelty'])

,pct_missing,auroc_1feat,fraud_rate_if_1,fraud_rate_if_0,pct_rows_1
feature,,,,,
fe_uid_new_device,90.4361,0.5331,0.0711,0.0956,31.6380
fe_uid_new_browser,87.6966,0.5346,0.0785,0.1057,33.4262
fe_uid_new_email,50.4481,0.5077,0.0373,0.0427,13.2175
fe_uid_new_product,38.6373,0.5066,0.0241,0.0408,3.1487
fe_uid_n_prior_devices,0.0000,0.5981,NaN,NaN,NaN
fe_uid_n_prior_emails,0.0000,0.5689,NaN,NaN,NaN
fe_uid_n_prior_browsers,0.0000,0.6107,NaN,NaN,NaN
fe_fp_n_prior_cards,78.9311,0.5428,NaN,NaN,NaN
fe_fp_n_prior_uids,78.9311,0.5315,NaN,NaN,NaN


## 5. Spending speed and escalation

Source: domain knowledge, tested in EDA 9.3, 9.4, 9.5 and 9.10. Bursts (9.4: 2.4% fraud with no earlier transaction in 24h vs 8.0% with 4 to 9; the 1-hour and 7-day counts rise
the same way) and repeated amounts (9.10) were supported, card testing (9.3) partly. The spending-spike idea (9.5)
was not supported.

Once a stolen card works, the fraudster usually tries to spend as much as possible before it gets blocked. That
shows up as bursts of transactions, so these features compare each transaction with the
customer's own history.

| Feature | Why we think it helps |
| --- | --- |
| `fe_uid_prior_count` | Number of earlier transactions. EDA 8: customers seen before have more fraud than first-seen ones (4.0% vs 2.7%), rising to 6.5% with 16 or more earlier transactions, which fits Vesta labelling a flagged customer's later transactions as fraud. |
| `fe_uid_secs_since_prev` | Time since the customer's last transaction. EDA 8: gaps under about 4.5 minutes have 8.5% fraud, falling to 1.0% for gaps over about 4 weeks. |
| `fe_uid_count_{1h, 24h, 7d}` | Number of transactions in the last hour, day and week. EDA 9.4: 1 hour 2.9% → 15.8% at 10+, 7 days 2.2% → 9.2%. |
| `fe_uid_spend_24h_rel_daily` | (spend in the last 24h + this amount) divided by the customer's average daily spend so far. EDA 9.5: the opposite of a spending spree; more than 5 times the daily average has 1.8% fraud vs 9.4% below half of it. |
| `fe_uid_small_then_large` | The previous amount was under $5 and this one is at least 10 times bigger: test the card, then use it. EDA 9.3: 10.6% fraud, but only 132 rows. |
| `fe_uid_repeat_amt_count` | How many times the customer has paid exactly this amount before. EDA 9.10 found repeats are riskier (3.6% → 6.3%), possibly scripted fraud. |

The time windows only count transactions before the current one (if two share a timestamp, only the one earlier in
the table counts). Together with the other `fe_uid_` features, these
form group G3, the customer-history aggregates (section 12).

One feature we left out on purpose is "was the customer's previous transaction fraud?". It would score very well
because of how Vesta labels customers, but it isn't realistic: chargebacks come in weeks later, so a real system
wouldn't know that label yet. It's also basically target encoding of `uid`, which `Context.md` rules out.

Considered but left out (EDA 9.4, 9.5): the amount compared with the customer's average, largest and previous
amount (`fe_uid_amt_rel_prior_mean`, `_rel_prior_max`, `_rel_prev`; fraud barely changes across bands, 3.2–4.9%),
and the amount spent in the last 24 hours (`fe_uid_spend_24h`; any spend raises fraud, but the amount adds little
beyond the 24-hour count).

In [8]:
signal_table(fams['velocity'])

,pct_missing,auroc_1feat,fraud_rate_if_1,fraud_rate_if_0,pct_rows_1
feature,,,,,
fe_uid_prior_count,0.0000,0.5738,NaN,NaN,NaN
fe_uid_secs_since_prev,38.6373,0.6732,NaN,NaN,NaN
fe_uid_small_then_large,38.6373,0.5004,0.1061,0.0403,0.0455
fe_uid_repeat_amt_count,0.0000,0.5463,NaN,NaN,NaN
fe_uid_count_1h,0.0000,0.5788,NaN,NaN,NaN
fe_uid_count_24h,0.0000,0.6231,NaN,NaN,NaN
fe_uid_spend_24h_rel_daily,38.6373,0.6548,NaN,NaN,NaN
fe_uid_count_7d,0.0000,0.6429,NaN,NaN,NaN


## 6. Email, browser and device details

| Feature | Source | Why we think it helps |
| --- | --- | --- |
| `fe_P_email_provider`, `fe_R_email_provider` | EDA 6, 6.2 | The ~60 email domains grouped into providers (google, yahoo, microsoft, apple, aol, anonymous, other). A few bigger groups are easier to learn from than lots of small domains. Recipient provider: Google 11.5% vs other 1.6%. |
| `fe_P_email_tld` | Domain knowledge, EDA 9.14 (cross-border idea not supported) | The end of the domain (com, net, mx, ...). The EDA found no cross-border pattern (`.mx` 1.6%, `.de` 0 of 399), but endings still differ: `.com` 3.7%, `.net` 1.8%, `.es` 9.3%. |
| `fe_email_anonymous` | Domain knowledge, EDA 9.6 (direction not supported) | Anonymous providers turned out *less* risky: anonymous.com and protonmail.com together 2.4% vs 3.7% for mainstream providers (protonmail.com alone 46%, but only 67 rows). Kept as a weak, stable signal. |
| `fe_email_match` | EDA 6 | Whether the purchaser and recipient domains match. Missing if either one is blank. Same domain has 9.3% fraud vs 2.8% for different domains. |
| `fe_browser_name` | EDA 6, 6.1 | Browser family from `id_31`. The browser is one of the strongest identity columns (spread 0.26 within identity rows), and EDA 12 recommends a family rather than the exact version, since 11.8% of validation rows have a version never seen in training. |
| `fe_browser_generic` | Domain knowledge, EDA 9.13 (supported) | "Generic" browser strings can point to bots or faked user agents: 9.0% fraud vs 7.4% for named browsers. |
| `fe_browser_versions_behind` | Domain knowledge, EDA 9.13 (supported) | How far behind the newest version of the same browser seen so far. Fraud setups sometimes use old or emulated browsers. |
| `fe_device_brand` | Domain knowledge, EDA 9.14 (supported) | Device brand from `DeviceInfo` (Motorola 16.2% and Samsung 11.5% fraud vs Apple 4.5%). |
| `fe_identity_os_missing` | EDA 6.1 and 11.1 | 1 if the row has an identity record but no OS (`id_30`), 0 if it has both, missing without an identity record. The raw OS and screen values changed meaning over time, but this state stayed the riskiest (7.5% → 13.8% fraud). |

Most of these just parse the row's own values. `fe_browser_versions_behind` uses a running maximum over earlier
rows. Considered but left out: `fe_devicetype_os_mismatch` (device type vs OS: EDA 9.7 found only 46 mismatched
rows, so the idea can't be tested, and its "reversed" result in section 10 mostly reflects whether the OS is
filled in), `fe_screen_aspect` (reversed), and `fe_os_family`, `fe_screen_w`, `fe_screen_h`, `fe_screen_area` (faded). These come from `id_30` and `id_33`, whose raw values are
excluded too (section 11).

In [9]:
display(signal_table(fams['identity']))
rate_by_level(fams['identity'], 'fe_P_email_provider')

,pct_missing,n_levels,auroc_1feat,fraud_rate_if_1,fraud_rate_if_0,pct_rows_1
feature,,,,,,
fe_P_email_provider,15.5760,7.0,NaN,NaN,NaN,NaN
fe_R_email_provider,75.9500,7.0,NaN,NaN,NaN,NaN
fe_P_email_tld,15.5760,9.0,NaN,NaN,NaN,NaN
fe_email_anonymous,15.5760,NaN,0.5128,0.0242,0.0371,7.5287
fe_email_match,77.5695,NaN,0.5672,0.0929,0.0282,81.0124
fe_browser_name,75.1365,42.0,NaN,NaN,NaN,NaN
fe_browser_generic,75.1365,NaN,0.5165,0.0894,0.0739,18.4731
fe_browser_versions_behind,80.0797,NaN,0.5005,NaN,NaN,NaN
fe_device_brand,78.9311,14.0,NaN,NaN,NaN,NaN


,n,fraud_rate
level,,
google,183103,0.0439
yahoo,84899,0.0226
<missing>,73586,0.0298
microsoft,48667,0.0519
anonymous,30028,0.0242
aol,22886,0.0234
other,22603,0.0204
apple,6660,0.0279


## 7. Location and card type

Source: EDA 3, 6 and 6.2.

| Feature | Why we think it helps |
| --- | --- |
| `fe_addr2_not_mode` | `addr2` looks like a country code, and one value covers about 99% of the training rows that have an `addr2` (88% of all rows; 11% are missing). Any other value most likely means a foreign billing address. EDA 6.2: 10.4% fraud vs 2.4%. |
| `fe_card3_not_mode` | `card3` has one value for about 88% of training rows. The other values may be cards issued abroad, though that's a guess since the column is masked. EDA 6.2: 11.2% vs 2.5%, 38% of all fraud. |
| `fe_card4_card6` | Card network and card type together (e.g. visa_debit, discover_credit). EDA 6.2: credit cards 6.7–7.6% except American Express credit (2.9%). Raw `card4` is excluded because its link to fraud reversed over time, but this combination stays stable. |

The most common values come from the training block. Considered but left out: `fe_uid_dist1_rel_prior_mean` (`dist1`
compared with the customer's past average; reversed).

In [10]:
display(signal_table(fams['geo_card']))
rate_by_level(fams['geo_card'], 'fe_card4_card6')

,pct_missing,auroc_1feat,fraud_rate_if_1,fraud_rate_if_0,pct_rows_1,n_levels
feature,,,,,,
fe_addr2_not_mode,11.3796,0.5152,0.1038,0.0243,0.9389,NaN
fe_card3_not_mode,0.1742,0.6334,0.1116,0.0249,11.8259,NaN
fe_card4_card6,0.0000,NaN,NaN,NaN,NaN,15.0


,n,fraud_rate
level,,
visa_debit,238826,0.0254
mastercard_debit,110604,0.0209
visa_credit,69303,0.0670
mastercard_credit,40228,0.0734
american express_credit,7077,0.0285
discover_credit,5106,0.0758
nan_nan,823,0.0292
discover_debit,288,0.0486
american express_debit,117,0.0342


## 8. Missing values and the M and C columns

Source: EDA 3 (missingness is a signal, and V columns go missing in blocks), 6 and 6.2 (M flags), 7 and 8.2 (C1,
C13).

In this dataset, whether a value is filled in tells you something. Identity data only exists for some merchants
and products, and whole blocks of V columns are present or missing depending on the type of transaction.

| Feature | Why we think it helps |
| --- | --- |
| `fe_n_missing` | Number of missing raw columns in the row, a rough summary of what data the merchant sent. |
| `fe_v_missing_pattern` | Which of the 14 V-column groups are missing (one column checked per group), stored as a pattern ID. Different patterns roughly line up with different products or checkout flows. |
| `fe_m_true`, `fe_m_false`, `fe_m_missing` | The M columns are match checks (e.g. does the name or address match the card). We count how many matched, didn't match, or weren't checked. EDA 6.2: no true flag 5.9% vs 1.6–3.0%. M4 has different values (M0/M1/M2), so it's left as it is. |
| `fe_uid_c1_rel_prior_mean`, `fe_uid_c13_rel_prior_mean` | The C columns count things linked to the card, like addresses or devices. EDA 8.2: C1 above twice its past average has 14.1% fraud vs 3.3%; for C13 a drop below half is the risky case (7.0% vs 3.6%). |

These are counts from the row itself, plus past-only averages for the C ratios.

In [11]:
signal_table(fams['missing'])

,pct_missing,auroc_1feat,n_levels
feature,,,
fe_n_missing,0.0000,0.6652,NaN
fe_v_missing_pattern,0.0000,NaN,209.0
fe_m_true,0.0000,0.6386,NaN
fe_m_false,0.0000,0.6091,NaN
fe_m_missing,0.0000,0.6385,NaN
fe_uid_c1_rel_prior_mean,38.6523,0.6049,NaN
fe_uid_c13_rel_prior_mean,41.6653,0.5340,NaN


## 9. How common a value is

For card, address, email domain, device, browser and amount (to the cent), we take the number of earlier rows with
the same value divided by the number of earlier rows. Source: domain knowledge, EDA 9.15. Rare values were riskier
for amounts (6.9% fraud in the rarest fifth vs 1.9% in the most common), devices (13.5% in the rarest fifth) and
browsers (11.7%), and slightly for billing regions (3.1% vs 2.3–2.4%). For card and email the expected direction
didn't hold (rare values are safer, e.g. rarest `card1` fifth 2.6%), but the signal is stable. The same share
for screen size (`fe_freq_id_33`) was considered but left out (reversed).

Computing these over the whole table would use future rows, so we only count earlier ones. The values are a bit
noisy at the very start of the data, when there aren't many earlier rows yet.

In [12]:
signal_table(fams['frequency'])

,pct_missing,auroc_1feat
feature,,
fe_freq_card1,0.0002,0.5168
fe_freq_addr1,11.3798,0.5279
fe_freq_P_emaildomain,15.5760,0.5532
fe_freq_DeviceInfo,78.9311,0.5888
fe_freq_id_31,75.1365,0.5842
fe_freq_TransactionAmt,0.0002,0.6328


## 10. Do the engineered features hold over time?

The same check as EDA 11.1, now on the engineered features (`fsel.time_stability`). For each feature on its own:
learn the fraud rate per value (or per quantile bin) on the first 30 days of the training block, score the last 30
days with it, and compute the AUC. The early signal is measured out of sample (two halves of the first 30 days,
each scored with rates learned on the other), because scoring the rows the rates were learned on makes noise look
like signal. Each feature lands in "holds later", "reversed later", "faded later" (clear signal early, none later),
"weak in both windows" or "constant early"; reversed and faded need 99.9% intervals.

A feature that reverses or fades would teach the model a rule that stops working on later months, so those are
left out. The second table runs the check on the 21 candidate features we considered and left out, rebuilt from the
raw columns: 11 reversed or faded; 3 (`fe_dow`, `fe_uid_new_os`, `fe_uid_new_screen`) showed no clear signal in
either window and had no EDA support either; and 7 hold over time but were left out because the EDA found no support for the idea behind
them: the 3 `fe_card1_` features (EDA 9.11: no steady rise, and `fe_card1_new_addr1` separates fraud only through
its missing state; section 4), the amount compared with the customer's average, largest and previous amount (EDA
9.5, section 5), and the 24-hour spend, whose signal is already carried by the kept 24-hour count (EDA 9.4). Of the kept features, only `fe_amt_lt5` is weak in
both windows; it is kept because it is 1 on only 0.2% of rows (amounts under $5), so a test over all rows can't judge
it, and EDA 9.3 shows its signal on those rows (6.5% fraud vs 3.5%).

In [13]:
fe_cols = [c for f in fams.values() for c in f.columns]
fe_train = pd.concat([df[['TransactionDay', 'isFraud']], *fams.values()], axis=1).iloc[:dfe.N_TRAIN]
stab_fe = fsel.time_stability(fe_train, fe_cols)
print(stab_fe['result'].value_counts().to_string())
assert not stab_fe['result'].isin(fsel.UNSTABLE).any(), stab_fe[stab_fe['result'].isin(fsel.UNSTABLE)]
stab_fe.sort_values('auc_late').round(3).head(10)

src/feature_selection.py:50: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return col_early.astype(object).fillna("(missing)"), col_late.astype(object).fillna("(missing)")
src/feature_selection.py:50: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return col_early.astype(object).fillna("(missing)"), col_late.astype(object).fillna("(missing)")
src/feature_selection.py:50: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to th

result
holds later             54
weak in both windows     1


,auc_early,auc_late,late_low,late_high,result
fe_amt_lt5,0.485,0.500,0.490,0.509,weak in both windows
fe_P_email_tld,0.521,0.515,0.505,0.525,holds later
fe_uid_repeat_amt_count,0.553,0.516,0.506,0.526,holds later
fe_uid_small_then_large,0.583,0.520,0.510,0.530,holds later
fe_uid_new_product,0.598,0.521,0.511,0.531,holds later
fe_uid_hour_dev,0.593,0.523,0.513,0.533,holds later
fe_uid_new_email,0.582,0.528,0.518,0.538,holds later
fe_email_anonymous,0.524,0.530,0.520,0.540,holds later
fe_uid_prior_count,0.614,0.538,0.528,0.548,holds later
fe_D4n,0.583,0.555,0.545,0.566,holds later


In [14]:
# Candidate features considered and left out, rebuilt here from the raw columns for the check
uid_key = uid.cat.codes.to_numpy(np.int64)
card_key = dfe.combine(df['card1'])
amt = df['TransactionAmt'].to_numpy(np.float64)
dt = df['TransactionDT'].to_numpy(np.int64)
ang = 2 * np.pi * ((dt % dfe.DAY) // dfe.HOUR) / 24
os_raw = df['id_30'].astype(object).str.lower()
os_fam = os_raw.str.extract(r'^(windows|ios|android|mac|linux)', expand=False).fillna('other').where(os_raw.notna())
scr = df['id_33'].astype(object).str.extract(r'^(\d+)x(\d+)$').astype('float64').replace(0, np.nan)
scr_w, scr_h = scr[0].where(scr[1].notna()).to_numpy(), scr[1].where(scr[0].notna()).to_numpy()
dtype_ = df['DeviceType'].astype(object)
mismatch = (((dtype_ == 'desktop') & os_fam.isin(['ios', 'android']))
            | ((dtype_ == 'mobile') & os_fam.isin(['windows', 'mac', 'linux'])))
id33 = df['id_33']
left_out = pd.DataFrame({
    'fe_amt_round_dollar': (fams['amount']['fe_amt_decimals'].to_numpy() == 0).astype(float),
    'fe_amt_round_10': np.isclose(amt % 10, 0, atol=1e-6).astype(float),
    'fe_dow': ((dt // dfe.DAY) % 7).astype(float),
    'fe_uid_hour_concentration': np.hypot(dfe.prior_mean(uid_key, np.sin(ang)), dfe.prior_mean(uid_key, np.cos(ang))),
    'fe_uid_new_os': dfe.new_value_flag(uid_key, df['id_30']),
    'fe_uid_new_screen': dfe.new_value_flag(uid_key, id33),
    'fe_os_family': os_fam.to_numpy(),
    'fe_screen_w': scr_w, 'fe_screen_h': scr_h, 'fe_screen_area': scr_w * scr_h,
    'fe_screen_aspect': dfe._ratio(np.fmax(scr_w, scr_h), np.fmin(scr_w, scr_h)),
    'fe_devicetype_os_mismatch': np.where(dtype_.notna() & os_fam.notna(), mismatch.astype(float), np.nan),
    'fe_uid_dist1_rel_prior_mean': dfe._ratio(df['dist1'], dfe.prior_mean(uid_key, df['dist1'])),
    'fe_freq_id_33': np.where(id33.notna(), dfe._ratio(dfe.prior_count(dfe.combine(id33)), np.arange(len(df))), np.nan),
    'fe_card1_new_addr1': dfe.new_value_flag(card_key, df['addr1']),
    'fe_card1_n_prior_addr1': dfe.distinct_prior(card_key, df['addr1'])[0],
    'fe_card1_n_prior_emails': dfe.distinct_prior(card_key, df['P_emaildomain'])[0],
    'fe_uid_amt_rel_prev': dfe._ratio(amt, dfe.prev_value(uid_key, amt)),
    'fe_uid_amt_rel_prior_mean': dfe._ratio(amt, dfe.prior_mean(uid_key, amt)),
    'fe_uid_amt_rel_prior_max': dfe._ratio(amt, dfe.prior_max(uid_key, amt)),
    'fe_uid_spend_24h': dfe.window_prior(uid_key, dt, amt, dfe.DAY)[1],
}, index=df.index)
LEFT_OUT = list(left_out.columns)
left_out_train = pd.concat([df[['TransactionDay', 'isFraud']], left_out], axis=1).iloc[:dfe.N_TRAIN]
stab_left_out = fsel.time_stability(left_out_train, LEFT_OUT)
del left_out, left_out_train
stab_left_out.round(3)

src/feature_selection.py:50: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return col_early.astype(object).fillna("(missing)"), col_late.astype(object).fillna("(missing)")
src/feature_selection.py:50: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return col_early.astype(object).fillna("(missing)"), col_late.astype(object).fillna("(missing)")
src/feature_selection.py:50: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to th

,auc_early,auc_late,late_low,late_high,result
fe_amt_round_dollar,0.537,0.507,0.497,0.517,faded later
fe_amt_round_10,0.539,0.480,0.470,0.489,reversed later
fe_dow,0.499,0.504,0.494,0.514,weak in both windows
fe_uid_hour_concentration,0.606,0.510,0.500,0.520,faded later
fe_uid_new_os,0.502,0.489,0.479,0.498,weak in both windows
fe_uid_new_screen,0.505,0.510,0.500,0.520,weak in both windows
fe_os_family,0.574,0.489,0.479,0.499,faded later
fe_screen_w,0.563,0.500,0.490,0.510,faded later
fe_screen_h,0.572,0.497,0.487,0.507,faded later
fe_screen_area,0.569,0.507,0.497,0.517,faded later


## 11. Raw columns used as model inputs

This follows EDA section 13, using the same rules, on training rows only:

| Excluded | Why | Evidence |
| --- | --- | --- |
| `TransactionID`, `TransactionDT`, `TransactionDay`, `uid` | Identifiers and raw time; `uid` is for grouping only | EDA 4, 5.2 |
| `id_12`, `id_27` | Add nothing beyond whether an identity record exists | EDA 6.1 |
| Redundant V columns | Within each group of V columns that go missing together, drop a column if \|r\| > 0.75 with one already kept | EDA 3; threshold from `notebooks/03_v_threshold.ipynb`: mean PR-AUC within noise of no pruning down to r = 0.70, lower in every fold from 0.65 down |
| Columns whose link to fraud reversed or faded over time | A model would learn a pattern that no longer holds | EDA 11.1, same check as section 10 |

All models should get the same input columns; how each model represents them (imputation, scaling, encoding) can
differ.

In [15]:
raw_cols = [c for c in df.columns if c not in dfe.ID_COLS and c != dfe.TARGET]
v_keep, v_drop = fsel.prune_v(train)
stab_raw = fsel.time_stability(train, raw_cols)
no_info = ['id_12', 'id_27']
unstable_raw = [c for c in raw_cols if stab_raw.loc[c, 'result'] in fsel.UNSTABLE
                and c not in set(v_drop) | set(no_info)]
excluded_raw = {
    'no added information (EDA 6.1)': no_info,
    f'redundant V columns, |r| > {fsel.V_THRESHOLD}': v_drop,
    'link to fraud reversed or faded over time (EDA 11.1)': unstable_raw,
}
dropped = {c for cols in excluded_raw.values() for c in cols}
raw_feature_cols = [c for c in raw_cols if c not in dropped]
is_v = lambda c: c.startswith('V') and c[1:].isdigit()
print(f'raw columns: {len(raw_cols)}; V kept by pruning: {len(v_keep)} of 339')
for k, v in excluded_raw.items():
    print(f'  excluded, {k}: {len(v)}')
print(f'raw model inputs: {len(raw_feature_cols)} ({sum(map(is_v, raw_feature_cols))} V); '
      f'engineered: {len(fe_cols)}; total: {len(raw_feature_cols) + len(fe_cols)}')
print('non-V columns excluded for stability:', ', '.join(c for c in unstable_raw if not is_v(c)))

src/feature_selection.py:50: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return col_early.astype(object).fillna("(missing)"), col_late.astype(object).fillna("(missing)")
src/feature_selection.py:50: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return col_early.astype(object).fillna("(missing)"), col_late.astype(object).fillna("(missing)")
src/feature_selection.py:50: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to th

raw columns: 432; V kept by pruning: 142 of 339
  excluded, no added information (EDA 6.1): 2
  excluded, redundant V columns, |r| > 0.75: 197
  excluded, link to fraud reversed or faded over time (EDA 11.1): 36
raw model inputs: 197 (113 V); engineered: 55; total: 252
non-V columns excluded for stability: card4, id_14, id_30, id_32, id_33, id_34, id_38


src/feature_selection.py:50: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return col_early.astype(object).fillna("(missing)"), col_late.astype(object).fillna("(missing)")
src/feature_selection.py:50: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return col_early.astype(object).fillna("(missing)"), col_late.astype(object).fillna("(missing)")


## 12. Feature groups for the experiments

The experiments add features in four cumulative groups, ordered by how much history a feature needs. Each step
answers one question:

- **G0 → G1:** does engineering a single transaction help?
- **G1 → G2:** does knowing how common the card, device and other values are help?
- **G2 → G3:** does a customer profile help? (RQ3 in `Context.md`)

| Group | Adds | What it needs at prediction time | Closest set in `Context.md` |
| --- | --- | --- | --- |
| G0 raw | the raw input columns from section 11 | the transaction only | F0 (same idea; raw list chosen in section 11) |
| G1 row-level | engineered from the transaction's own values (sections 1, 2, 3, 6, 7, 8) | the transaction only | (extra step) |
| G2 card and device history | past-only counts over the device fingerprint and all earlier rows, including how common this card, address or amount is (sections 4, 6, 9) | earlier transactions | F1 (same role, different contents) |
| G3 customer history | past-only aggregates over `uid` (sections 2, 4, 5, 8) | earlier transactions of the same customer | F2 (same role, different contents) |

A brand-new customer only gets useful values from G0 to G2, which ties in with reporting results separately for
returning and new customers. G2 and G3 play the roles of F1 and F2 but don't copy them: only some of the
card1 and uid aggregates listed for F1/F2 are here, chosen by the EDA checks, plus others the EDA supported.
The table below lists every engineered feature with its group.

In [16]:
# Feature groups: what each step of the ladder adds (G0 raw -> G3 customer history)
G2_HISTORY = ['fe_freq_card1', 'fe_freq_addr1', 'fe_freq_P_emaildomain', 'fe_freq_DeviceInfo', 'fe_freq_id_31',
              'fe_freq_TransactionAmt', 'fe_fp_n_prior_cards', 'fe_fp_n_prior_uids', 'fe_fp_prior_count',
              'fe_browser_versions_behind']
G3_CUSTOMER = [c for c in fe_cols if c.startswith('fe_uid_')]
G1_ROW = [c for c in fe_cols if c not in set(G2_HISTORY) | set(G3_CUSTOMER)]
feature_groups = {
    'G0_raw': raw_feature_cols,
    'G1_row_level': G1_ROW,
    'G2_card_device_history': G2_HISTORY,
    'G3_customer_history': G3_CUSTOMER,
}
assert sorted(G1_ROW + G2_HISTORY + G3_CUSTOMER) == sorted(fe_cols)
names = list(feature_groups)
feature_sets = {f'G{i}': [c for g in names[:i + 1] for c in feature_groups[g]] for i in range(len(names))}
for g, cols in feature_groups.items():
    print(f'{g}: adds {len(cols)}, cumulative {len(feature_sets[g[:2]])}')

group_of = {c: g for g in ['G1_row_level', 'G2_card_device_history', 'G3_customer_history'] for c in feature_groups[g]}
family_of = {c: fam for fam, f in fams.items() for c in f.columns}
group_table = (pd.DataFrame({'group': [group_of[c] for c in fe_cols], 'family': [family_of[c] for c in fe_cols],
                             'feature': fe_cols})
               .sort_values(['group', 'family']).groupby(['group', 'family'])['feature']
               .agg(lambda s: ', '.join(s)).to_frame())
with pd.option_context('display.max_colwidth', 200):
    display(group_table)

G0_raw: adds 197, cumulative 197
G1_row_level: adds 27, cumulative 224
G2_card_device_history: adds 10, cumulative 234
G3_customer_history: adds 18, cumulative 252


feature
group                  family                                                                                                                                                                                 
G1_row_level           amount                                                                                                                            fe_amt_log, fe_amt_cents, fe_amt_decimals, fe_amt_lt5
                       d_norm                                                                                                                                        fe_D1n, fe_D4n, fe_D10n, fe_D11n, fe_D15n
                       geo_card                                                                                                                           fe_addr2_not_mode, fe_card3_not_mode, fe_card4_card6
                       identity     fe_P_email_provider, fe_R_email_provider, fe_P_email_tld, fe_email_anonymous, fe_email_match, fe_browser_name, fe_browser_generic, fe_device_brand, fe_identity_os_missing
                       missing                                                                                                         fe_n_missing, fe_v_missing_pattern, fe_m_true, fe_m_false, fe_m_missing
                       time                                                                                                                                                                            fe_hour
G2_card_device_history frequency                                                                fe_freq_card1, fe_freq_addr1, fe_freq_P_emaildomain, fe_freq_DeviceInfo, fe_freq_id_31, fe_freq_TransactionAmt
                       identity                                                                                                                                                     fe_browser_versions_behind
                       novelty                                                                                                                      fe_fp_n_prior_cards, fe_fp_n_prior_uids, fe_fp_prior_count
G3_customer_history    missing                                                                                                                             fe_uid_c1_rel_prior_mean, fe_uid_c13_rel_prior_mean
                       novelty                             fe_uid_new_device, fe_uid_new_browser, fe_uid_new_email, fe_uid_new_product, fe_uid_n_prior_devices, fe_uid_n_prior_emails, fe_uid_n_prior_browsers
                       time                                                                                                                                                                    fe_uid_hour_dev
                       velocity   fe_uid_prior_count, fe_uid_secs_since_prev, fe_uid_small_then_large, fe_uid_repeat_amt_count, fe_uid_count_1h, fe_uid_count_24h, fe_uid_spend_24h_rel_daily, fe_uid_count_7d

## 13. Putting it together and saving

The output has every raw column from the joined table, `uid`, and all the `fe_` columns, in the original time
order, so excluded raw columns are still there for analysis; the model input lists are in the YAML file. The splits
are based on row position, so the table must not be re-sorted. Numbers are stored as `float32` and text as
`category`, the same as the rest of the repo.

`config/features_v2.yaml` lists the raw input columns, the engineered columns by family, the excluded raw columns
with the reason, the feature groups and sets from section 12, and the constants learned from the training block.

In [17]:
full = dfe.assemble(df, uid, fams)

assert len(full) == len(df) and (full['TransactionID'].to_numpy() == df['TransactionID'].to_numpy()).all()
assert full['TransactionDT'].is_monotonic_increasing
assert full.columns.is_unique
assert not any(full[c].dtype == object for c in full.columns)
print(f'{full.shape[0]:,} rows x {full.shape[1]} columns ({len(fe_cols)} engineered); '
      f'{full.memory_usage(deep=True).sum() / 1024**3:.2f} GB in memory')

590,540 rows x 492 columns (55 engineered); 1.04 GB in memory


In [18]:
feature_list = {
    'version': 'features_v2',
    'parquet': str(dfe.FEATURES_PATH.relative_to(dfe.ROOT)),
    'built_by': 'notebooks/02_feature_engineering.ipynb (logic in src/domain_features.py, src/feature_selection.py)',
    'target': dfe.TARGET,
    'not_features': {
        'TransactionID': 'row identifier',
        'TransactionDT': 'raw time; trends with time, so a model would extrapolate poorly on later blocks',
        'TransactionDay': 'same as TransactionDT',
        'uid': 'customer identifier for grouping and returning-vs-new analysis; raw ID would be memorised',
    },
    'train_block_rows': dfe.N_TRAIN,
    'constants_fit_on_train_block': consts,
    'v_threshold': fsel.V_THRESHOLD,
    'raw_feature_cols': raw_feature_cols,
    'excluded_raw_cols': excluded_raw,
    'engineered_cols': {name: list(f.columns) for name, f in fams.items()},
    'considered_not_used': LEFT_OUT,
    'feature_groups': feature_groups,
    'feature_sets': feature_sets,
    'categorical_cols': [c for c in raw_feature_cols + fe_cols
                         if isinstance(full[c].dtype, pd.CategoricalDtype)],
    'notes': [
        'V columns: pruned at r = 0.75, agreed by the group (notebooks/03_v_threshold.ipynb).',
        'Fit imputers, scalers and encoders in the model notebook on training rows only (inside each CV fold).',
        'Test block (last 10% of rows) stays unscored until the experiment freeze.',
    ],
}
assert len(set(raw_feature_cols) & set(fe_cols)) == 0

dfe.FEATURES_PATH.parent.mkdir(parents=True, exist_ok=True)
full.to_parquet(dfe.FEATURES_PATH, index=False)
with open(dfe.FEATURE_LIST_PATH, 'w') as f:
    yaml.safe_dump(feature_list, f, sort_keys=False, width=110)

print('saved', dfe.FEATURES_PATH.relative_to(dfe.ROOT),
      f'({dfe.FEATURES_PATH.stat().st_size / 1024**2:.0f} MB) and', dfe.FEATURE_LIST_PATH.relative_to(dfe.ROOT))

saved data/processed/features_v2.parquet (113 MB) and config/features_v2.yaml


## 14. All engineered features at a glance

Every new feature sorted by its single-feature AUROC on the training block. Same caveat as before: this is a
sanity check, not how we pick features.

In [19]:
overview = pd.concat({name: signal_table(f) for name, f in fams.items()}, names=['family', 'feature'])
overview.sort_values('auroc_1feat', ascending=False)

pct_missing  auroc_1feat  fraud_rate_if_1  fraud_rate_if_0  pct_rows_1  n_levels
family    feature                                                                                                     
velocity  fe_uid_secs_since_prev          38.6373       0.6732              NaN              NaN         NaN       NaN
missing   fe_n_missing                     0.0000       0.6652              NaN              NaN         NaN       NaN
velocity  fe_uid_spend_24h_rel_daily      38.6373       0.6548              NaN              NaN         NaN       NaN
          fe_uid_count_7d                  0.0000       0.6429              NaN              NaN         NaN       NaN
missing   fe_m_true                        0.0000       0.6386              NaN              NaN         NaN       NaN
          fe_m_missing                     0.0000       0.6385              NaN              NaN         NaN       NaN
d_norm    fe_D15n                         16.4606       0.6348              NaN              NaN         NaN       NaN
geo_card  fe_card3_not_mode                0.1742       0.6334           0.1116           0.0249     11.8259       NaN
frequency fe_freq_TransactionAmt           0.0002       0.6328              NaN              NaN         NaN       NaN
d_norm    fe_D10n                         14.0530       0.6277              NaN              NaN         NaN       NaN
          fe_D1n                           0.1981       0.6272              NaN              NaN         NaN       NaN
identity  fe_identity_os_missing          74.5013       0.6249           0.1142           0.0437     45.1380       NaN
velocity  fe_uid_count_24h                 0.0000       0.6231              NaN              NaN         NaN       NaN
novelty   fe_uid_n_prior_browsers          0.0000       0.6107              NaN              NaN         NaN       NaN
missing   fe_m_false                       0.0000       0.6091              NaN              NaN         NaN       NaN
d_norm    fe_D4n                          29.5861       0.6050              NaN              NaN         NaN       NaN
missing   fe_uid_c1_rel_prior_mean        38.6523       0.6049              NaN              NaN         NaN       NaN
novelty   fe_uid_n_prior_devices           0.0000       0.5981              NaN              NaN         NaN       NaN
d_norm    fe_D11n                         52.0653       0.5948              NaN              NaN         NaN       NaN
frequency fe_freq_DeviceInfo              78.9311       0.5888              NaN              NaN         NaN       NaN
          fe_freq_id_31                   75.1365       0.5842              NaN              NaN         NaN       NaN
velocity  fe_uid_count_1h                  0.0000       0.5788              NaN              NaN         NaN       NaN
          fe_uid_prior_count               0.0000       0.5738              NaN              NaN         NaN       NaN
novelty   fe_uid_n_prior_emails            0.0000       0.5689              NaN              NaN         NaN       NaN
identity  fe_email_match                  77.5695       0.5672           0.0929           0.0282     81.0124       NaN
amount    fe_amt_cents                     0.0000       0.5606              NaN              NaN         NaN       NaN
frequency fe_freq_P_emaildomain           15.5760       0.5532              NaN              NaN         NaN       NaN
amount    fe_amt_decimals                  0.0000       0.5503              NaN              NaN         NaN       NaN
velocity  fe_uid_repeat_amt_count          0.0000       0.5463              NaN              NaN         NaN       NaN
novelty   fe_fp_n_prior_cards             78.9311       0.5428              NaN              NaN         NaN       NaN
          fe_uid_new_browser              87.6966       0.5346           0.0785           0.1057     33.4262       NaN
missing   fe_uid_c13_rel_prior_mean       41.6653       0.5340              NaN              NaN         NaN   

## Loading the file in a model notebook

```python
import pandas as pd, yaml
spec = yaml.safe_load(open('../config/features_v2.yaml'))
full = pd.read_parquet('../' + spec['parquet'])
features = spec['feature_sets']['G3']        # or 'G0', 'G1', 'G2' for the smaller sets
X, y = full[features], full[spec['target']]
# split by row position: train = rows [0, 472432); validation and test follow (see Context.md)
```

Things to keep in mind:

- Split by row position. The table is already in time order, so don't shuffle it.
- Tune with time-based CV inside the training block, use the validation block only for scoring and choosing the
  threshold (never for early stopping), and leave the test block alone until the freeze. The model notebooks (04
  to 09) do all of this through `src/experiment.py`.
- XGBoost can use the category columns directly with `enable_categorical=True` and `tree_method='hist'`. Otherwise,
  convert them to integer codes.
- For LR and the MLP, fit imputation, scaling and encoding on the training rows only.
- Log every run to `results/log.csv` and note `features_v2` in the params.